In [1]:
from transformers import RobertaTokenizer, T5ForConditionalGeneration

tokenizer = RobertaTokenizer.from_pretrained('Salesforce/codet5-small')
model = T5ForConditionalGeneration.from_pretrained('Salesforce/codet5-small')

text = "def greet(user): print(f'hello <extra_id_0>!')"
input_ids = tokenizer(text, return_tensors="pt").input_ids

# simply generate a single sequence
generated_ids = model.generate(input_ids, max_length=10)
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))
# this prints "user: {user.name}"


tokenizer_config.json:   0%|          | 0.00/1.48k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/703k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/294k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/12.5k [00:00<?, ?B/s]

/opt/conda/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/242M [00:00<?, ?B/s]

user: {user.name}


In [2]:
from transformers import RobertaTokenizer, T5ForConditionalGeneration
from torch.utils.data import Dataset, DataLoader
import torch

# Load tokenizer
tokenizer = RobertaTokenizer.from_pretrained('Salesforce/codet5-small')

# Custom Dataset for Verilog to C translation
class VerilogToCCodeDataset(Dataset):
    def __init__(self, verilog_file, c_code_file, tokenizer, max_length=512):
        with open(verilog_file, 'r') as vf:
            self.verilog_lines = vf.readlines()
        with open(c_code_file, 'r') as cf:
            self.c_code_lines = cf.readlines()

        # Ensure both files have the same number of lines
        assert len(self.verilog_lines) == len(self.c_code_lines), "Mismatch in number of Verilog and C code lines"
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.verilog_lines)

    def __getitem__(self, idx):
        verilog_code = self.verilog_lines[idx].strip()
        c_code = self.c_code_lines[idx].strip()

        # Tokenize Verilog (source) and C (target)
        input_encoding = self.tokenizer(
            verilog_code,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        target_encoding = self.tokenizer(
            c_code,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        # Prepare the input and target for training
        return {
            'input_ids': input_encoding['input_ids'].squeeze(),
            'attention_mask': input_encoding['attention_mask'].squeeze(),
            'labels': target_encoding['input_ids'].squeeze()
        }

# Example paths to your dataset
verilog_file = "/kaggle/input/code-verilog-to-c/train.src"
c_code_file = "/kaggle/input/code-verilog-to-c/train.tgt"

# Create dataset and dataloader
dataset = VerilogToCCodeDataset(verilog_file, c_code_file, tokenizer)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)


In [3]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, T5ForConditionalGeneration
import torch

# Check GPU availability
print("Is CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("Number of GPUs:", torch.cuda.device_count())

# Load pre-trained CodeT5 model
model = T5ForConditionalGeneration.from_pretrained('Salesforce/codet5-small')

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Model is on GPU:", next(model.parameters()).is_cuda)

# Define training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    save_total_limit=2,
    save_strategy="epoch",
    predict_with_generate=True,
    # Add verbose logging
    logging_strategy="steps",  # Log at every step
    logging_steps=100,  # Log every 100 steps
    logging_first_step=True,  # Log the first step
    report_to=["none"],  # Disable wandb logging
)

# Trainer for fine-tuning
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    eval_dataset = dataset
)

# Fine-tune the model
trainer.train()

Is CUDA available: True
GPU Name: Tesla P100-PCIE-16GB
Number of GPUs: 1
Model is on GPU: True


/opt/conda/lib/python3.10/site-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,0.625900,0.224476
2,0.245600,0.183166
3,0.215500,0.173032


TrainOutput(global_step=516, training_loss=0.32361733589985575, metrics={'train_runtime': 701.7524, 'train_samples_per_second': 5.878, 'train_steps_per_second': 0.735, 'total_flos': 558284931072000.0, 'train_loss': 0.32361733589985575, 'epoch': 3.0})

In [4]:
from transformers import RobertaTokenizer, T5ForConditionalGeneration
from torch.utils.data import Dataset, DataLoader
import torch
from tqdm import tqdm
from nltk.translate.bleu_score import corpus_bleu
import nltk
nltk.download('punkt')

# Load tokenizer and model
tokenizer = RobertaTokenizer.from_pretrained('Salesforce/codet5-small')
# model = T5ForConditionalGeneration.from_pretrained('./results')  # Load your fine-tuned model

# Custom Dataset for testing
class TestDataset(Dataset):
    def __init__(self, src_file, tgt_file, tokenizer, max_length=512):
        with open(src_file, 'r') as sf, open(tgt_file, 'r') as tf:
            self.src_lines = sf.readlines()
            self.tgt_lines = tf.readlines()
        assert len(self.src_lines) == len(self.tgt_lines), "Mismatch in number of source and target lines"
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.src_lines)

    def __getitem__(self, idx):
        src = self.src_lines[idx].strip()
        tgt = self.tgt_lines[idx].strip()
        return src, tgt

# Load test data
test_dataset = TestDataset('/kaggle/input/code-verilog-to-c/test.src', 
                           '/kaggle/input/code-verilog-to-c/test.tgt', 
                           tokenizer)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Generate predictions
predictions = []
references = []

with torch.no_grad():
    for src, tgt in tqdm(test_loader, desc="Generating predictions"):
        input_ids = tokenizer(src[0], return_tensors="pt", max_length=512, truncation=True, padding="max_length").input_ids.to(device)
        output = model.generate(input_ids, max_length=512, num_beams=4, early_stopping=True)
        decoded_output = tokenizer.decode(output[0], skip_special_tokens=True)
        predictions.append(decoded_output.split())
        references.append([tgt[0].split()])

# Calculate BLEU score
bleu_score = corpus_bleu(references, predictions)
print(f"BLEU Score: {bleu_score}")

# Print some example translations
num_examples = 5
print("\nExample Translations:")
for i in range(min(num_examples, len(test_dataset))):
    print(f"\nExample {i + 1}:")
    print(f"Source (Verilog): {test_dataset[i][0]}")
    print(f"Target (C): {test_dataset[i][1]}")
    print(f"Prediction: {' '.join(predictions[i])}")

# Optional: Save predictions to a file
with open('predictions.txt', 'w') as f:
    for pred in predictions:
        f.write(' '.join(pred) + '\n')

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


/opt/conda/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Generating predictions: 100%|██████████| 5/5 [00:07<00:00,  1.57s/it]

BLEU Score: 0.47303864384080546

Example Translations:

Example 1:
Source (Verilog): module entry_point ( , ap_start, , ap_done, , ap_idle, , ap_ready, , speed, , time_r, , temptemp1, , temptemp1_ap_vld, , ap_return , ); , input ap_start; , output ap_done; , output ap_idle; , output ap_ready; , input [31:0] speed; , input [31:0] time_r; , output [31:0] temptemp1; , output temptemp1_ap_vld; , output [31:0] ap_return; , reg temptemp1_ap_vld; , wire [31:0] distance_fu_45_p2; , wire ap_ce_reg; , entry_point_mul_32s_32s_32_1_1 #( , .ID( 1 ), , .NUM_STAGE( 1 ), , .din0_WIDTH( 32 ), , .din1_WIDTH( 32 ), , .dout_WIDTH( 32 )) , mul_32s_32s_32_1_1_U1( , .din0(time_r), , .din1(speed), , .dout(distance_fu_45_p2) , ); , always @ (*) begin , if ((ap_start == 1'b1)) begin , temptemp1_ap_vld = 1'b1; , end else begin , temptemp1_ap_vld = 1'b0; , end , end , assign ap_done = ap_start; , assign ap_idle = 1'b1; , assign ap_ready = ap_start; , assign ap_return = 32'd0; , assign temptemp1 = distance_fu_45_p